# Spam Email Classifier — Naive Bayes & SVM
**Author:** Pritam Kumar | NIT Hamirpur

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

## 1. Load Dataset
> Uses the SMS Spam Collection dataset from UCI (downloaded via URL).

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df['label'].value_counts())
df['label'].value_counts().plot(kind='bar', color=['steelblue','tomato'])
plt.title('Class Distribution'); plt.xticks(rotation=0); plt.show()

df['msg_len'] = df['message'].apply(len)
df.groupby('label')['msg_len'].hist(bins=40, alpha=0.7)
plt.title('Message Length by Class'); plt.legend(['ham','spam']); plt.show()

## 3. Preprocess

In [ ]:
df['label_enc'] = (df['label'] == 'spam').astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label_enc'], test_size=0.2, random_state=42, stratify=df['label_enc']
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 4. Model 1 — Naive Bayes

In [ ]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=10000)),
    ('clf', MultinomialNB(alpha=0.1)),
])
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}%")
print(classification_report(y_test, y_pred_nb, target_names=['ham','spam']))

## 5. Model 2 — Linear SVM

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=10000)),
    ('clf', LinearSVC(C=1.0, max_iter=1000)),
])
svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
print(classification_report(y_test, y_pred_svm, target_names=['ham','spam']))

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_pred, title in zip(axes, [y_pred_nb, y_pred_svm], ['Naive Bayes', 'SVM']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['ham','spam'], yticklabels=['ham','spam'])
    ax.set_title(f'{title} Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

## 7. ROC Curve (Naive Bayes)

In [ ]:
y_proba_nb = nb_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba_nb)
auc = roc_auc_score(y_test, y_proba_nb)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'NB AUC = {auc:.3f}', color='steelblue')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve — Naive Bayes'); plt.legend(); plt.show()

## 8. Try Your Own Message

In [ ]:
def predict_spam(text, model=svm_pipeline):
    pred = model.predict([text])[0]
    return "🚨 SPAM" if pred == 1 else "✅ HAM"

samples = [
    "Congratulations! You've won a free iPhone. Click now!",
    "Hey, are we still on for lunch tomorrow?",
    "URGENT: Your bank account has been compromised. Call now!",
    "Can you send me the notes from today's lecture?"
]
for s in samples:
    print(f"{predict_spam(s)}  →  {s[:60]}")